In [ ]:
# Generated from dataops/01-monitoring.yaml -- do not edit by hand.
#
# Evaluates every declared expectation and SLA against what actually landed,
# then applies the enforcement mode for this environment.
#
# The rules live in ttfabric.monitoring; this notebook only supplies the tables
# and decides what to do with the verdicts.

from datetime import datetime, timezone
import uuid

from pyspark.sql import functions as F
from ttfabric.monitoring import (
    run_expectations, evaluate_slas, FAIL, ERROR, WARN,
)

# Set by the pipeline; defaults to dev so a manual run is never the strict one.
environment = "dev"
try:
    environment = str(env).strip() or "dev"      # noqa: F821  (notebook parameter)
except NameError:
    pass

EXPECTATIONS = [{'table': 'bronze_commerce_customers',
  'layer': 'bronze',
  'checks': [{'id': 'BR-CUST-001',
              'rule': 'schema_match',
              'severity': 'critical',
              'registry_ref': 'commerce.customers'},
             {'id': 'BR-CUST-002',
              'rule': 'row_count_delta',
              'severity': 'warning',
              'max_increase_pct': 50,
              'max_decrease_pct': 20},
             {'id': 'BR-CUST-003',
              'rule': 'freshness',
              'severity': 'error',
              'column': '_ingested_at',
              'max_age_hours': 26}],
  'expected_columns': ['customer_id',
                       'first_name',
                       'last_name',
                       'email',
                       'phone',
                       'region',
                       'created_at',
                       'updated_at']},
 {'table': 'bronze_commerce_orders',
  'layer': 'bronze',
  'checks': [{'id': 'BR-ORD-001',
              'rule': 'schema_match',
              'severity': 'critical',
              'registry_ref': 'commerce.orders'},
             {'id': 'BR-ORD-002',
              'rule': 'row_count_delta',
              'severity': 'warning',
              'max_increase_pct': 50,
              'max_decrease_pct': 20},
             {'id': 'BR-ORD-003',
              'rule': 'freshness',
              'severity': 'error',
              'column': '_ingested_at',
              'max_age_hours': 26}],
  'expected_columns': ['order_id',
                       'customer_id',
                       'order_date',
                       'total_price',
                       'status',
                       'items_count',
                       'created_at',
                       'updated_at']},
 {'table': 'bronze_commerce_order_items',
  'layer': 'bronze',
  'checks': [{'id': 'BR-ITEM-001',
              'rule': 'schema_match',
              'severity': 'critical',
              'registry_ref': 'commerce.order_items'},
             {'id': 'BR-ITEM-002',
              'rule': 'row_count_delta',
              'severity': 'warning',
              'max_increase_pct': 50,
              'max_decrease_pct': 20}],
  'expected_columns': ['order_item_id',
                       'order_id',
                       'product_id',
                       'quantity',
                       'unit_price',
                       'subtotal',
                       'created_at']},
 {'table': 'bronze_commerce_products',
  'layer': 'bronze',
  'checks': [{'id': 'BR-PROD-001',
              'rule': 'schema_match',
              'severity': 'critical',
              'registry_ref': 'commerce.products'}],
  'expected_columns': ['product_id',
                       'product_name',
                       'category',
                       'price',
                       'stock',
                       'created_at',
                       'updated_at']},
 {'table': 'stg_customers',
  'layer': 'silver',
  'checks': [{'id': 'SL-CUST-001',
              'rule': 'not_null',
              'column': 'email',
              'severity': 'warning',
              'warn_above': 0.022,
              'fail_above': 0.04,
              'measured_on': 'input',
              'note': 'Calibrated from customer_null_email = 29/2048 = 1.42%'},
             {'id': 'SL-CUST-002',
              'rule': 'unique',
              'columns': ['email'],
              'severity': 'warning',
              'warn_above': 0.035,
              'fail_above': 0.06,
              'measured_on': 'input',
              'note': 'Calibrated from customer_duplicate = 48/2048 = 2.34%'},
             {'id': 'SL-CUST-003',
              'rule': 'unique',
              'columns': ['email'],
              'severity': 'critical',
              'fail_above': 0.0,
              'measured_on': 'output'},
             {'id': 'SL-CUST-004',
              'rule': 'not_null',
              'column': 'region',
              'severity': 'error',
              'fail_above': 0.0,
              'measured_on': 'output'},
             {'id': 'SL-CUST-005',
              'rule': 'accepted_values',
              'column': 'region',
              'severity': 'error',
              'values': ['North', 'South', 'East', 'West', 'Central'],
              'fail_above': 0.0,
              'measured_on': 'output'}],
  'source': 'bronze_commerce_customers',
  'expected_columns': ['customer_id',
                       'email',
                       'first_name',
                       'last_name',
                       'full_name',
                       'phone',
                       'region',
                       'created_at',
                       'updated_at']},
 {'table': 'stg_products',
  'layer': 'silver',
  'checks': [{'id': 'SL-PROD-001',
              'rule': 'range',
              'column': 'list_price',
              'min': 0.01,
              'max': 100000,
              'severity': 'warning',
              'warn_above': 0.02,
              'fail_above': 0.05,
              'measured_on': 'input',
              'note': 'Calibrated from product_invalid_price = 3/300 = 1.0%'},
             {'id': 'SL-PROD-002',
              'rule': 'range',
              'column': 'list_price',
              'min': 0.01,
              'max': 100000,
              'severity': 'critical',
              'fail_above': 0.0,
              'measured_on': 'output'},
             {'id': 'SL-PROD-003',
              'rule': 'not_null',
              'column': 'category',
              'severity': 'error',
              'fail_above': 0.0,
              'measured_on': 'output',
              'note': "Defaulted to 'Uncategorised', so output nulls are impossible"}],
  'source': 'bronze_commerce_products',
  'expected_columns': ['product_id',
                       'product_name',
                       'category',
                       'list_price',
                       'stock',
                       'created_at',
                       'updated_at']},
 {'table': 'stg_order_items',
  'layer': 'silver',
  'checks': [{'id': 'SL-ITEM-001',
              'rule': 'arithmetic_consistency',
              'severity': 'warning',
              'expression': 'subtotal = quantity * unit_price',
              'tolerance': 0.01,
              'warn_above': 0.03,
              'fail_above': 0.05,
              'measured_on': 'input',
              'note': 'Calibrated from item_subtotal_mismatch = 1494/74782 = 2.0%'},
             {'id': 'SL-ITEM-002',
              'rule': 'arithmetic_consistency',
              'severity': 'critical',
              'expression': 'subtotal = quantity * unit_price',
              'tolerance': 0.01,
              'fail_above': 0.0,
              'measured_on': 'output'},
             {'id': 'SL-ITEM-003',
              'rule': 'referential_integrity',
              'column': 'product_id',
              'references': 'stg_products.product_id',
              'severity': 'error',
              'fail_above': 0.0,
              'measured_on': 'output'},
             {'id': 'SL-ITEM-004',
              'rule': 'range',
              'column': 'quantity',
              'min': 1,
              'max': 1000,
              'severity': 'error',
              'fail_above': 0.0,
              'measured_on': 'output'}],
  'source': 'bronze_commerce_order_items',
  'expected_columns': ['order_item_id',
                       'order_id',
                       'product_id',
                       'quantity',
                       'unit_price',
                       'subtotal',
                       'subtotal_as_reported',
                       'subtotal_was_corrected',
                       'created_at']},
 {'table': 'stg_orders',
  'layer': 'silver',
  'checks': [{'id': 'SL-ORD-001',
              'rule': 'unique',
              'columns': ['order_id'],
              'severity': 'warning',
              'warn_above': 0.038,
              'fail_above': 0.065,
              'measured_on': 'input',
              'note': 'Calibrated from order_duplicate = 646/25646 = 2.5%'},
             {'id': 'SL-ORD-002',
              'rule': 'unique',
              'columns': ['order_id'],
              'severity': 'critical',
              'fail_above': 0.0,
              'measured_on': 'output'},
             {'id': 'SL-ORD-003',
              'rule': 'referential_integrity',
              'column': 'customer_id',
              'references': 'stg_customers.customer_id',
              'severity': 'warning',
              'warn_above': 0.016,
              'fail_above': 0.03,
              'measured_on': 'input',
              'note': 'Calibrated from order_orphan_customer = 268/25646 = 1.04%'},
             {'id': 'SL-ORD-004',
              'rule': 'referential_integrity',
              'column': 'customer_id',
              'references': 'stg_customers.customer_id',
              'severity': 'critical',
              'fail_above': 0.0,
              'measured_on': 'output'},
             {'id': 'SL-ORD-005',
              'rule': 'not_null',
              'column': 'order_date',
              'severity': 'warning',
              'warn_above': 0.013,
              'fail_above': 0.025,
              'measured_on': 'input',
              'note': 'Calibrated from order_null_date = 224/25646 = 0.87%'},
             {'id': 'SL-ORD-006',
              'rule': 'arithmetic_consistency',
              'severity': 'warning',
              'expression': 'order_total = sum(order_items.subtotal)',
              'tolerance': 0.01,
              'warn_above': 0.021,
              'fail_above': 0.04,
              'measured_on': 'input',
              'note': 'Calibrated from order_total_mismatch = 359/25646 = 1.40%'},
             {'id': 'SL-ORD-007',
              'rule': 'range',
              'column': 'order_total',
              'min': 0.01,
              'max': 10000000,
              'severity': 'error',
              'fail_above': 0.0,
              'measured_on': 'output'},
             {'id': 'SL-ORD-008',
              'rule': 'accepted_values',
              'column': 'status',
              'severity': 'error',
              'values': ['pending', 'confirmed', 'shipped', 'delivered', 'cancelled'],
              'fail_above': 0.0,
              'measured_on': 'output'}],
  'source': 'bronze_commerce_orders',
  'expected_columns': ['order_id',
                       'customer_id',
                       'order_date',
                       'order_date_key',
                       'order_total',
                       'order_total_as_reported',
                       'order_total_was_corrected',
                       'status',
                       'is_revenue',
                       'items_count_as_reported',
                       'items_count',
                       'created_at',
                       'updated_at']},
 {'table': 'dbo.fct_sales',
  'layer': 'gold',
  'checks': [{'id': 'GD-SALES-001',
              'rule': 'unique',
              'columns': ['order_item_id'],
              'severity': 'critical',
              'fail_above': 0.0,
              'note': 'Enforces the declared grain'},
             {'id': 'GD-SALES-002',
              'rule': 'not_null',
              'column': 'customer_sk',
              'severity': 'critical',
              'fail_above': 0.0},
             {'id': 'GD-SALES-003',
              'rule': 'not_null',
              'column': 'product_sk',
              'severity': 'critical',
              'fail_above': 0.0},
             {'id': 'GD-SALES-004',
              'rule': 'not_null',
              'column': 'order_date_sk',
              'severity': 'critical',
              'fail_above': 0.0},
             {'id': 'GD-SALES-005',
              'rule': 'arithmetic_consistency',
              'severity': 'critical',
              'expression': 'sum(line_revenue) = silver.sum(stg_order_items.subtotal)',
              'tolerance': 0.01,
              'fail_above': 0.0,
              'note': 'End-to-end reconciliation: gold must tie back to silver'}]},
 {'table': 'dbo.dim_customer',
  'layer': 'gold',
  'checks': [{'id': 'GD-CUST-001',
              'rule': 'unique',
              'columns': ['customer_sk'],
              'severity': 'critical',
              'fail_above': 0.0},
             {'id': 'GD-CUST-002',
              'rule': 'unique',
              'columns': ['customer_id'],
              'severity': 'critical',
              'filter': 'is_current = TRUE',
              'fail_above': 0.0,
              'note': 'Overlapping validity windows are the usual SCD2 bug'}]},
 {'table': 'dbo.dim_product',
  'layer': 'gold',
  'checks': [{'id': 'GD-PROD-001',
              'rule': 'unique',
              'columns': ['product_sk'],
              'severity': 'critical',
              'fail_above': 0.0},
             {'id': 'GD-PROD-002',
              'rule': 'unique',
              'columns': ['product_id'],
              'severity': 'critical',
              'filter': 'is_current = TRUE',
              'fail_above': 0.0}]}]

SLAS = [{'table': 'dbo.fct_sales',
  'freshness_hours': 2,
  'completeness_pct': 99.5,
  'accuracy_pct': 99.9,
  'availability_pct': 99.9},
 {'table': 'bi.vw_sales_summary',
  'freshness_hours': 2,
  'completeness_pct': 99.5,
  'availability_pct': 99.9},
 {'table': 'stg_customers', 'freshness_hours': 26, 'completeness_pct': 98.0}]

ENFORCEMENT = {'dev': {'mode': 'warn', 'block_on': []},
 'qa': {'mode': 'block', 'block_on': ['error', 'critical']},
 'uat': {'mode': 'block', 'block_on': ['error', 'critical']},
 'prod': {'mode': 'alert', 'block_on': []}}

# Where each table lives. The monitor spans three storage items of two
# different kinds; nothing else in the pipeline crosses all of them at once.
LOCATIONS = {'bronze_commerce_customers': ('lakehouse', 'lh_bronze'),
 'bronze_commerce_orders': ('lakehouse', 'lh_bronze'),
 'bronze_commerce_order_items': ('lakehouse', 'lh_bronze'),
 'bronze_commerce_products': ('lakehouse', 'lh_bronze'),
 'stg_customers': ('lakehouse', 'lh_silver'),
 'stg_products': ('lakehouse', 'lh_silver'),
 'stg_order_items': ('lakehouse', 'lh_silver'),
 'stg_orders': ('lakehouse', 'lh_silver'),
 'fct_sales': ('warehouse', 'wh_gold'),
 'dim_customer': ('warehouse', 'wh_gold'),
 'dim_product': ('warehouse', 'wh_gold'),
 'vw_sales_summary': ('warehouse', 'wh_gold')}
DEFAULT_LAKEHOUSE = 'lh_silver'

RESULTS_TABLE = 'dq_run_log_monitor'
run_id = f"mon_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"

print(f"environment {environment}   run {run_id}")
print(f"{len(EXPECTATIONS)} tables, "
      f"{sum(len(e['checks']) for e in EXPECTATIONS)} checks, {len(SLAS)} SLAs")
print()


def read(name):
    """Resolve a table name to a DataFrame, wherever that layer actually lives.

    The monitor spans all three layers, and they are NOT in one place:

      bronze   lh_bronze    a different lakehouse -- must be qualified
      silver   lh_silver    this notebook's default -- unqualified works
      gold     wh_gold      a WAREHOUSE, which spark.read.table cannot reach
                            at all; it needs the Fabric DW connector

    Reading everything unqualified silently resolves against the default
    lakehouse, so every bronze and gold check fails as TABLE_OR_VIEW_NOT_FOUND
    -- which looks like missing data rather than a wrong lookup.
    """
    bare = name.rsplit(".", 1)[-1]
    kind, item = LOCATIONS.get(bare, ("lakehouse", None))

    if kind == "warehouse":
        # Lazy import: the connector registers spark.read.synapsesql as a side
        # effect, and is unavailable until it has been imported at least once.
        import com.microsoft.spark.fabric                  # noqa: F401
        from com.microsoft.spark.fabric.Constants import Constants  # noqa: F401
        schema = name.rsplit(".", 2)[-2] if "." in name else "dbo"
        return spark.read.synapsesql(f"{item}.{schema}.{bare}")   # noqa: F821

    if item and item != DEFAULT_LAKEHOUSE:
        return spark.read.table(f"{item}.{bare}")      # noqa: F821
    return spark.read.table(bare)                          # noqa: F821


# Previous row counts, so row_count_delta has something to compare against.
# Absent on a first run, which the rule reports as "not measurable" rather
# than as a breach.
previous_rows = {}
try:
    history = spark.read.table(RESULTS_TABLE)             # noqa: F821
    latest = history.filter(F.col("rule_id") == "row_count_delta") \
        .groupBy("table_name").agg(F.max("run_id").alias("run_id"))
    for row in history.join(latest, ["table_name", "run_id"]).collect():
        if row["rows"] is not None:
            previous_rows[row["table_name"]] = row["rows"]
except Exception as exc:
    print(f"no monitoring history yet ({type(exc).__name__}); "
          f"row_count_delta will be skipped on this run")

run = run_expectations(read, EXPECTATIONS, previous_rows=previous_rows)
run.results.extend(evaluate_slas(read, SLAS))

for result in run.results:
    print(result)

counts = run.counts()
print()
print("  ".join(f"{status}={count}" for status, count in sorted(counts.items())))

# --- persist -------------------------------------------------------------
rows = [(run_id, r.check_id, r.table, r.layer, r.rule, r.severity, r.status,
         float(r.measured) if r.measured is not None else None,
         float(r.limit) if r.limit is not None else None,
         int(r.rows) if r.rows is not None else None,
         r.detail, datetime.now(timezone.utc))
        for r in run.results]

schema = ("run_id string, check_id string, table_name string, layer string, "
          "rule_id string, severity string, status string, measured double, "
          "limit_value double, rows bigint, detail string, checked_at timestamp")
# mergeSchema: this is an append-only log, and adding a column to it should not
# be what stops monitoring from running. Delta otherwise rejects the whole write
# on any schema difference.
spark.createDataFrame(rows, schema).write.mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(RESULTS_TABLE)                            # noqa: F821
print(f"\n{len(rows)} results appended to {RESULTS_TABLE}")

# --- enforce -------------------------------------------------------------
policy = ENFORCEMENT.get(environment, {"mode": "warn", "block_on": []})
blocking = run.blocking(policy.get("block_on") or [])

print(f"\nenforcement for {environment}: mode={policy.get('mode')} "
      f"block_on={policy.get('block_on') or 'nothing'}")

if blocking:
    for result in blocking:
        print(f"  BLOCKING  {result}")
    # Raised so the pipeline stops. In an environment whose block_on is empty
    # this never fires, which is the point of declaring it per environment.
    raise RuntimeError(
        f"{len(blocking)} blocking breach(es) in {environment}: "
        + ", ".join(r.check_id for r in blocking))

breached = [r for r in run.results if r.status in (FAIL, ERROR, WARN)]
if breached:
    print(f"\n{len(breached)} breach(es) recorded, none blocking in {environment}.")
else:
    print("\nall checks passed")
